# 04 — Clustering pracowników HR

Segmentacja 1470 pracowników metodą **KMeans (k=4)** na podstawie 21 cech (kariera, wynagrodzenie, satysfakcja, mobilność, feature engineering v3).

Cel: zrozumieć naturalne grupy pracowników i powiązać je z ryzykiem attrition.

> Importuje biblioteki do clusteringu (KMeans, PCA, StandardScaler, silhouette_score), ładuje dane HR i generuje cechy inżynierowane.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
(ROOT / 'reports').mkdir(parents=True, exist_ok=True)

from data_loader import load_hr
from features import engineer_features

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

df = engineer_features(load_hr())
print(f'Zaladowano {len(df):,} pracownikow, {df.shape[1]} kolumn po feature engineering')

## 1. Cechy do clusteringu

Wybieramy 21 interpretowalnych cech numerycznych obejmujących:
- karierę i wynagrodzenie (Age, MonthlyIncome, JobLevel, TotalWorkingYears)
- staż i stabilność (YearsAtCompany, YearsInCurrentRole, YearsWithCurrManager)
- satysfakcję (JobSatisfaction, WorkLifeBalance, AvgSatisfaction)
- mobilność i obciążenie (NumCompaniesWorked, HighOvertime, DistanceFromHome)
- cechy v3 (BurnoutRiskScore, StabilityComposite, CareerGrowthIndex)

> Wybiera 21 interpretowalnych cech numerycznych (kariera, wynagrodzenie, satysfakcja, mobilność) i standaryzuje je do Z-score za pomocą StandardScaler.

In [ ]:
CLUSTER_FEATURES = [
    'Age', 'MonthlyIncome', 'JobLevel', 'TotalWorkingYears',
    'YearsAtCompany', 'YearsInCurrentRole', 'YearsWithCurrManager',
    'YearsSinceLastPromotion',
    'JobSatisfaction', 'WorkLifeBalance', 'EnvironmentSatisfaction',
    'RelationshipSatisfaction', 'AvgSatisfaction',
    'NumCompaniesWorked', 'HighOvertime', 'DistanceFromHome',
    'BurnoutRiskScore', 'StabilityComposite', 'CareerGrowthIndex',
    'CompaniesPerYear', 'IncomePerJobLevel',
]

X_raw = df[CLUSTER_FEATURES].copy()
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)
print(f'Macierz cech: {X.shape} (standaryzowana)')
X_raw.describe().T[['mean', 'std', 'min', 'max']].round(2).head(10)

## 2. Dobór liczby klastrów — metoda łokcia + silhouette

> Testuje k=2..7 klastrów — oblicza inercję (metoda łokcia) i Silhouette score dla każdego k, rysuje wykresy pomocnicze do wyboru optymalnego k.

In [ ]:
ks = range(2, 8)
inertias, sil_scores = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, labels, sample_size=1000, random_state=42))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(list(ks), inertias, 'o-', color='steelblue')
ax1.set_xlabel('Liczba klastrów k')
ax1.set_ylabel('Inercja (WCSS)')
ax1.set_title('Metoda łokcia')
ax2.plot(list(ks), sil_scores, 's-', color='crimson')
ax2.set_xlabel('Liczba klastrów k')
ax2.set_ylabel('Silhouette score')
ax2.set_title('Silhouette score')
fig.suptitle('Dobór k — pracownicy HR', fontsize=12)
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'clustering_elbow.png', dpi=100, bbox_inches='tight')
plt.show()

print('Silhouette scores:')
for k, s in zip(ks, sil_scores):
    print(f'  k={k}: {s:.3f}')
print(f'\nNajlepszy silhouette: k={list(ks)[int(np.argmax(sil_scores))]}')
print('Wybieramy k=4 dla bogatszej interpretowalnosci biznesowej.')

## 3. KMeans k=4 — segmentacja

> Trenuje KMeans z k=4 i przypisuje każdego z 1470 pracowników do jednego z 4 klastrów; wyświetla liczebność i attrition rate per klaster.

In [ ]:
K = 4
km = KMeans(n_clusters=K, random_state=42, n_init=20)
df['Cluster'] = km.fit_predict(X)

print('Rozklad klastrów:')
for c in sorted(df['Cluster'].unique()):
    n = (df['Cluster'] == c).sum()
    attr = df[df['Cluster'] == c]['Attrition'].mean()
    print(f'  Klaster {c}: {n} pracownikow ({n/len(df):.0%}), Attrition={attr:.1%}')

## 4. Profile klastrów — średnie cech

> Oblicza profile klastrów jako średnie wartości 16 kluczowych cech (wiek, dochód, satysfakcja, staż itd.) per klaster.

In [ ]:
profile_cols = [
    'Age', 'MonthlyIncome', 'JobLevel', 'TotalWorkingYears',
    'YearsAtCompany', 'YearsSinceLastPromotion',
    'JobSatisfaction', 'WorkLifeBalance', 'AvgSatisfaction',
    'NumCompaniesWorked', 'HighOvertime', 'DistanceFromHome',
    'BurnoutRiskScore', 'StabilityComposite', 'CareerGrowthIndex',
    'Attrition',
]
profiles = df.groupby('Cluster')[profile_cols].mean().round(2)
profiles['n'] = df['Cluster'].value_counts().sort_index()
display(profiles)

> Rysuje heatmapę Z-score profili klastrów — kolor czerwony oznacza wartość powyżej, zielony poniżej globalnej średniej dla danej cechy.

In [ ]:
# Heatmapa profili (Z-score wzgledem sredniej globalnej)
heat_cols = [
    'Age', 'MonthlyIncome', 'JobLevel', 'TotalWorkingYears',
    'YearsAtCompany', 'YearsSinceLastPromotion',
    'JobSatisfaction', 'WorkLifeBalance',
    'NumCompaniesWorked', 'HighOvertime', 'BurnoutRiskScore',
    'Attrition',
]
global_mean = df[heat_cols].mean()
global_std  = df[heat_cols].std()
profiles_z  = (profiles[heat_cols] - global_mean) / global_std

fig, ax = plt.subplots(figsize=(14, 3.5))
sns.heatmap(
    profiles_z[heat_cols],
    annot=profiles[heat_cols].round(2),
    fmt='.2f',
    cmap='RdYlGn_r',
    center=0,
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Z-score vs. srednia'},
    annot_kws={'size': 8},
)
ax.set_title('Profile klastrów — Z-score (czerwony=powyzej sredniej, zielony=ponizej)', pad=10)
ax.set_ylabel('Klaster')
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'clustering_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Wizualizacja PCA 2D

> Redukuje 21 cech do 2D metodą PCA i rysuje scatter plot pracowników — raz kolorując klastry, raz kolorując Attrition (0/1).

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)
var = pca.explained_variance_ratio_

COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Klastry
ax = axes[0]
for c in sorted(df['Cluster'].unique()):
    mask = df['Cluster'] == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=COLORS[c], alpha=0.45, s=18,
               label=f'Klaster {c} (n={mask.sum()})')
ax.set_xlabel(f'PC1 ({var[0]:.1%} wariancji)')
ax.set_ylabel(f'PC2 ({var[1]:.1%} wariancji)')
ax.set_title('Klastry pracowników (PCA 2D)')
ax.legend(fontsize=9)

# Attrition
ax = axes[1]
colors_attr = ['#66c2a5' if a == 0 else '#fc8d62' for a in df['Attrition']]
ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors_attr, alpha=0.35, s=18)
p0 = mpatches.Patch(color='#66c2a5', label='Zostaje (0)')
p1 = mpatches.Patch(color='#fc8d62', label='Rezygnacja (1)')
ax.set_xlabel(f'PC1 ({var[0]:.1%} wariancji)')
ax.set_ylabel(f'PC2 ({var[1]:.1%} wariancji)')
ax.set_title('Attrition w przestrzeni PCA')
ax.legend(handles=[p0, p1], fontsize=9)

fig.suptitle('Clustering HR — PCA 2D', fontsize=12)
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'clustering_pca.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'PCA wyjaśnia {sum(var):.1%} wariancji')

## 6. Statystyki attrition i wynagrodzenia wg klastra

> Oblicza statystyki per klaster (attrition rate, średni dochód, średnia satysfakcja, odsetek nadgodzin) i rysuje trzy wykresy słupkowe z liniami średniej globalnej.

In [ ]:
cluster_stats = df.groupby('Cluster').agg(
    n=('Attrition', 'count'),
    attrition_rate=('Attrition', 'mean'),
    avg_income=('MonthlyIncome', 'mean'),
    avg_age=('Age', 'mean'),
    avg_satisfaction=('AvgSatisfaction', 'mean'),
    pct_overtime=('HighOvertime', 'mean'),
).round(3)
display(cluster_stats)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
x = [str(c) for c in cluster_stats.index]
bar_colors = [COLORS[c] for c in cluster_stats.index]

axes[0].bar(x, cluster_stats['attrition_rate'], color=bar_colors, edgecolor='white')
axes[0].axhline(df['Attrition'].mean(), color='crimson', ls='--', lw=1.5,
                label=f"Srednia={df['Attrition'].mean():.1%}")
axes[0].set_title('Attrition rate wg klastra')
axes[0].set_ylabel('P(Attrition=1)')
axes[0].legend(fontsize=8)
for i, v in enumerate(cluster_stats['attrition_rate']):
    axes[0].text(i, v + 0.005, f'{v:.1%}', ha='center', fontsize=9, fontweight='bold')

axes[1].bar(x, cluster_stats['avg_income'], color=bar_colors, edgecolor='white')
axes[1].axhline(df['MonthlyIncome'].mean(), color='crimson', ls='--', lw=1.5)
axes[1].set_title('Srednie wynagrodzenie ($/mies.)')
for i, v in enumerate(cluster_stats['avg_income']):
    axes[1].text(i, v + 50, f'${v:.0f}', ha='center', fontsize=8)

axes[2].bar(x, cluster_stats['avg_satisfaction'], color=bar_colors, edgecolor='white')
axes[2].axhline(df['AvgSatisfaction'].mean(), color='crimson', ls='--', lw=1.5)
axes[2].set_title('Srednia satysfakcja (1-4)')
axes[2].set_ylim(0, 4)
for i, v in enumerate(cluster_stats['avg_satisfaction']):
    axes[2].text(i, v + 0.03, f'{v:.2f}', ha='center', fontsize=9)

fig.suptitle(f'Statystyki klastrów pracowników (KMeans k={K})', fontsize=12)
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'clustering_stats.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Opisy klastrów w języku naturalnym

Na podstawie profili (średnie cech względem globalnej średniej) każdy klaster otrzymuje interpretację biznesową.

---

### Klaster 0 — "Nomadzi" (30% zespołu, Attrition ~19%)

**Profil:** Pracownicy w średnim wieku (~36 lat) z **bardzo wysoką mobilnością** (średnio 5 firm w karierze). Krótki staż w firmie (~3 lata), pomimo czego wynagrodzenie i poziom stanowiska zbliżone do średniej. Niedawno awansowani.

**Interpretacja:** To pracownicy z przyzwyczajeniem do zmiany pracodawcy — nie z frustracji, ale ze strategicznego budowania kariery. Ryzyko odejścia wyższe niż średnia, ale nie dramatyczne. Motywuje ich rozwój, nie stabilność.

**Zalecenie HR:** Jasna ścieżka kariery, projekty dające nowe kompetencje, mentoring. Programy lojalnościowe mają ograniczony efekt.

---

### Klaster 1 — "Starterzy" (29% zespołu, Attrition ~22% — najwyższe!)

**Profil:** **Najmłodsi** pracownicy (~30 lat), **najkrótszy staż** (~3 lata), **najniższe wynagrodzenie** (~3 380 $/mies.), najniższy poziom stanowiska. Niska mobilność (1-2 firmy w karierze) — to często pierwsza lub druga praca.

**Interpretacja:** Klasyczna grupa wysokiego ryzyka: młodzi, niedostatecznie wynagradzani, z ograniczonymi możliwościami awansu i słabym dopasowaniem do stanowiska. Wypalenie zawodowe może rozwinąć się szybko.

**Zalecenie HR:** Podwyżki dla najniżej opłacanych ról, przyspieszony program awansów, buddy/mentoring od starszych pracowników, regularne check-iny z managerem.

---

### Klaster 2 — "Lojalni Stagnujący" (28% zespołu, Attrition ~10%)

**Profil:** Pracownicy z **długim stażem** w firmie (~10 lat), wynagrodzenie i poziom stanowiska zbliżone do średniej. Dawno bez awansu (~4 lata). Niska mobilność historyczna.

**Interpretacja:** Lojalni, stabilni pracownicy — filar organizacji. Niska rotacja, ale ryzyko **stagnacji i wypalenia** długoterminowego. Mogą czuć się "utknięci" bez perspektyw.

**Zalecenie HR:** Lateral moves (zmiana roli bez awansu), projekty specjalne, programy uznaniowe. Niebezpieczny moment: ~rok 12-15, kiedy stagnacja może stać się frustracja.

---

### Klaster 3 — "Seniorzy" (13% zespołu, Attrition ~9% — najniższe)

**Profil:** **Najstarsi i najbardziej doświadczeni** (~47 lat, 25 lat doświadczenia), **najwyższe wynagrodzenie** (~15 490 $/mies.), wysoki poziom stanowiska (menagerski/dyrektorski). Długi staż w firmie (~17 lat), ale dawno bez awansu (~6 lat).

**Interpretacja:** Kadra seniorska — stabilna, wysoko opłacana, z ograniczonymi powodami do odejścia. Ryzyko odejścia niskie, ale warto zadbać o sukcesję wiedzy i engagement strategiczny.

**Zalecenie HR:** Programy mentoringu odwróconego, projekty strategiczne, elastyczne formy pracy. Kluczowe dla retencji wiedzy organizacyjnej.

> Tworzy skondensowaną tabelę podsumowującą z nazwami klastrów (Nomadzi, Starterzy, Lojalni Stagnujący, Seniorzy), liczebnością, ryzykiem attrition i zalecanymi akcjami HR.

In [ ]:
# Podsumowanie — tabela opisow
cluster_summary = pd.DataFrame({
    'Nazwa': ['Nomadzi', 'Starterzy', 'Lojalni Stagnujacy', 'Seniorzy'],
    'N': [447, 431, 407, 185],
    'Udzial': ['30%', '29%', '28%', '13%'],
    'Attrition': ['~19%', '~22%', '~10%', '~9%'],
    'Ryzyko': ['Srednie', 'WYSOKIE', 'Niskie', 'Niskie'],
    'Kluczowa_cecha': [
        'Wysoka mobilnosc (5 firm)',
        'Mlody wiek + niskie zarobki',
        'Dlugi staz + brak awansu',
        'Doswiadczeni seniorzy',
    ],
    'Glowna_akcja_HR': [
        'Sciezka kariery + nowe projekty',
        'Podwyzki + mentoring',
        'Lateral moves + uznanie',
        'Sukcesja wiedzy',
    ],
}, index=[0, 1, 2, 3])
cluster_summary.index.name = 'Klaster'
display(cluster_summary)

> Zapisuje profile klastrów i tabelę podsumowującą do plików CSV: `reports/cluster_profiles.csv` i `reports/cluster_summary.csv`.

In [ ]:
# Zapis wynikow
profiles.to_csv(ROOT / 'reports' / 'cluster_profiles.csv')
cluster_summary.to_csv(ROOT / 'reports' / 'cluster_summary.csv')
print('Zapisano: cluster_profiles.csv, cluster_summary.csv')
print('Wykresy: clustering_elbow.png, clustering_pca.png, clustering_heatmap.png, clustering_stats.png')

## 8. Wnioski

| Klaster | Nazwa | Attrition | Priorytet HR |
|---|---|---|---|
| 0 | Nomadzi | 19% | Sredni |
| 1 | Starterzy | **22%** | **Wysoki** |
| 2 | Lojalni Stagnujacy | 10% | Niski |
| 3 | Seniorzy | 9% | Niski |

**Kluczowe obserwacje:**
1. Klaster 1 (Starterzy) to **priorytet interwencji HR** — najwyższy attrition, najniższe zarobki, najmłodsi
2. Klaster 0 (Nomadzi) wymaga **innego podejścia** niż retencja klasyczna — tych pracowników motywuje rozwój, nie stabilność
3. Klastry 2 i 3 są stabilne, ale wymagają profilaktyki długoterminowej (stagnacja, sukcesja wiedzy)
4. Łącznie klastry 0+1 (59% zespołu) generują **większość kosztów attrition**

**Ograniczenia:**
- Silhouette score dla k=4 (0.105) jest umiarkowany — granice klastrów nie są ostre
- KMeans zakłada klastry kuliste — w rzeczywistości profile mogą się nakładać
- Wyniki wrażliwe na dobór cech i normalizację